In [40]:
import pandas as pd
import numpy as np
import datetime

In [72]:
#read previous 90 days file
pastdf = pd.read_csv('/Users/emersonfrost/Desktop/caml_90days_previous.csv')

In [73]:
print(pastdf.shape)
pastdf.head(5)

(194235, 18)


,system:index,NIR,SWIR,blue,cloud_score,cyanobacteria_abundance,green,latitude,longitude,ndci,ndti,ndvi,ndwi,red,red_edge,sample_date,satellite_date,.geo
0,0_20200404T170851_20200404T171737_T14SQJ,0.303042,0.338892,0.276812,0.784902,6.061,0.278933,39.141931,-95.479569,0.061518,-0.005008,0.036407,-0.041403,0.281921,0.318489,2020-06-29,2020-04-04,"{""type"":""MultiPoint"",""coordinates"":[]}"
1,0_20200404T170851_20200404T171737_T15STD,0.301742,0.338304,0.274842,0.723308,6.061,0.277107,39.141931,-95.479569,0.063106,-0.005193,0.037172,-0.042355,0.280170,0.317642,2020-06-29,2020-04-04,"{""type"":""MultiPoint"",""coordinates"":[]}"
2,0_20200409T170849_20200409T172046_T14SQJ,0.190471,0.242455,0.179102,0.559030,6.061,0.186306,39.141931,-95.479569,0.114497,0.040531,0.049144,-0.008665,0.173056,0.216059,2020-06-29,2020-04-09,"{""type"":""MultiPoint"",""coordinates"":[]}"
3,0_20200409T170849_20200409T172046_T15STD,0.189511,0.241976,0.178344,0.545731,6.061,0.185647,39.141931,-95.479569,0.114474,0.040177,0.047975,-0.007842,0.172519,0.215444,2020-06-29,2020-04-09,"{""type"":""MultiPoint"",""coordinates"":[]}"
4,0_20200414T170851_20200414T171907_T14SQJ,0.079389,0.095053,0.046439,0.150185,6.061,0.066794,39.141931,-95.479569,-0.114757,0.001956,0.051338,-0.049359,0.071396,0.069393,2020-06-29,2020-04-14,"{""type"":""MultiPoint"",""coordinates"":[]}"


In [74]:
#cleaning
#general cleaning

#removing null values / those that didn't populate
print('before removing nulls', pastdf.shape)
pastdf = pastdf[(pastdf['green'] != -999) | (pastdf['blue'] != -999) | (pastdf['SWIR'] != -999) | (pastdf['NIR'] != -999)]
print('removed nulls', pastdf.shape)

#drop duplicate entries & take one with less cloud coverage
pastdf = (pastdf.sort_values('cloud_score', ascending = True).drop_duplicates(subset = ['cyanobacteria_abundance','sample_date', 'satellite_date']))
print('removed duplicates:', pastdf.shape)

before removing nulls (194235, 18)
removed nulls (193946, 18)
removed duplicates: (150380, 18)


In [75]:
#create multiple datasets with different cloud coverages 
#20% 
df_under20 = pastdf[pastdf['cloud_score'] <= 0.2]
print("Under 20%: ", df_under20.shape)
#30%
df_under30 = pastdf[pastdf['cloud_score'] <= 0.3]
print("Under 30%: ", df_under30.shape)
#50% 
df_under50 = pastdf[pastdf['cloud_score'] <= 0.5]
print("Under 50%: ", df_under50.shape)
#70%
df_under70 = pastdf[pastdf['cloud_score'] <= 0.7]
print("Under 70%: ", df_under70.shape)

Under 20%:  (65798, 18)
Under 30%:  (75522, 18)
Under 50%:  (92792, 18)
Under 70%:  (106343, 18)


In [76]:
#printing out the amount of data we keep
print(df_under20.shape[0] / pastdf.shape[0])
print(df_under30.shape[0] / pastdf.shape[0])
print(df_under50.shape[0] / pastdf.shape[0])
print(df_under70.shape[0] / pastdf.shape[0])

0.4375448862880702
0.5022077403910095
0.6170501396462296
0.707161856629871


In [77]:
#determining what the best number of timeseries datapoints per clouyd

#ie how many with 9, 8, 7 --> used to determine final sample size + shape of input
a_list = [df_under20, df_under30, df_under50, df_under70]
names = ['20%', '30%', '50%', '70%']
for i in range(len(a_list)):
    print(names[i])
    counts = (
        a_list[i].groupby(['sample_date', 'latitude', 'longitude', 'cyanobacteria_abundance'])
          .size()
          .reset_index(name='n_obs')
    )
    total = len(counts)
    print(f"Total unique samples: {total}")
    n_7 = (counts['n_obs'] >= 7).sum()
    n_8 = (counts['n_obs'] >= 8).sum()
    n_9 = (counts['n_obs'] >= 9).sum()

    print("Number of samples")
    print(f"7: {n_7} 8: {n_8} 9: {n_9}")
    print("Percentage of total unique samples")
    print(f"7: {n_7 / total} 8: {n_8 / total} 9: {n_9 / total}")

20%
Total unique samples: 7750
Number of samples
7: 4085 8: 3492 9: 3054
Percentage of total unique samples
7: 0.5270967741935484 8: 0.4505806451612903 9: 0.39406451612903226
30%
Total unique samples: 7818
Number of samples
7: 4787 8: 4231 9: 3750
Percentage of total unique samples
7: 0.6123049373241238 8: 0.5411870043489383 9: 0.47966231772831924
50%
Total unique samples: 7892
Number of samples
7: 5893 8: 5398 9: 4857
Percentage of total unique samples
7: 0.746705524581855 8: 0.6839837810440953 9: 0.6154333502280791
70%
Total unique samples: 7904
Number of samples
7: 6325 8: 5977 9: 5620
Percentage of total unique samples
7: 0.8002277327935222 8: 0.7561993927125507 9: 0.7110323886639676


In [85]:
#30 percent df --> 7 time points (4787, ~61%)
print("30% Dataframe w/ 7 time points per location")
cloud_30_7 = (
    df_under30.sort_values('satellite_date', ascending=False)
      .groupby(['latitude', 'longitude', 'cyanobacteria_abundance', 'sample_date'])
        .filter(lambda x: len(x) >= 7)
        .groupby(['latitude', 'longitude', 'cyanobacteria_abundance', 'sample_date'])
        .head(7)
)
print(f"df shape: {cloud_30_7.shape}")
print(f"unique locations: {cloud_30_7.shape[0] / 7}")

30% Dataframe w/ 7 time points per location
df shape: (33509, 18)
unique locations: 4787.0


In [86]:
#50 percent df --> 8 time points (5398, ~68%)
print("50% Dataframe w/ 8 time points per location")
cloud_50_8 = (
    df_under50.sort_values('satellite_date', ascending=False)
      .groupby(['latitude', 'longitude', 'cyanobacteria_abundance', 'sample_date'])
        .filter(lambda x: len(x) >= 8)
        .groupby(['latitude', 'longitude', 'cyanobacteria_abundance', 'sample_date'])
        .head(8)
)
print(f"df shape: {cloud_50_8.shape}")
print(f"unique locations: {cloud_50_8.shape[0] / 8}")

50% Dataframe w/ 8 time points per location
df shape: (43184, 18)
unique locations: 5398.0


In [87]:
#70 percent df --> 9 time points (5620, ~71%)
print("70% Dataframe w/ 9 time points per location")
cloud_70_9 = (
    df_under70.sort_values('satellite_date', ascending=False)
      .groupby(['latitude', 'longitude', 'cyanobacteria_abundance', 'sample_date'])
        .filter(lambda x: len(x) >= 9)
        .groupby(['latitude', 'longitude', 'cyanobacteria_abundance', 'sample_date'])
        .head(9)
)
print(f"df shape: {cloud_70_9.shape}")
print(f"unique locations: {cloud_70_9.shape[0] / 9}")

70% Dataframe w/ 9 time points per location
df shape: (50580, 18)
unique locations: 5620.0


In [88]:
print(cloud_30_7.columns)
print(cloud_50_8.columns)
print(cloud_70_9.columns)

Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo'],
      dtype='object')
Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo'],
      dtype='object')
Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo'],
      dtype='object')


In [89]:
#adding calculated columns
#function to get calculated columns
def add_calc_cols(df):
    #calculating statistics
    group_cols = ['latitude', 'longitude', 'sample_date', 'cyanobacteria_abundance']

    def compute_time_features(group):
        group = group.sort_values('satellite_date')
    
        #create list of differences within each sample
        diffs = group['satellite_date'].diff().dt.days.dropna()
    
        #calculating the total difference
        tot_diff = (group['sample_date'].iloc[0] - group['satellite_date'].iloc[0]).days
    
        return pd.Series({
            'mean_diff_days': diffs.mean(),
            'std_diff_days': diffs.std()
        })

    time_features = df.groupby(group_cols).apply(compute_time_features).reset_index()
    return time_features

In [90]:
#make everything time series
cloud_30_7['sample_date'] = pd.to_datetime(cloud_30_7['sample_date'])
cloud_30_7['satellite_date'] = pd.to_datetime(cloud_30_7['satellite_date'])

cloud_50_8['sample_date'] = pd.to_datetime(cloud_50_8['sample_date'])
cloud_50_8['satellite_date'] = pd.to_datetime(cloud_50_8['satellite_date'])

cloud_70_9['sample_date'] = pd.to_datetime(cloud_70_9['sample_date'])
cloud_70_9['satellite_date'] = pd.to_datetime(cloud_70_9['satellite_date'])

In [91]:
#calculated columns
#30%
print(cloud_30_7.shape)
cloud_30_time = add_calc_cols(cloud_30_7)
cloud_30_7 = cloud_30_7.merge(cloud_30_time, on= ['latitude', 'longitude', 'sample_date', 'cyanobacteria_abundance'])
print(cloud_30_7.shape)

(33509, 18)
(33509, 20)


/var/folders/9z/ltjvr1610w9g431sszz26tr40000gn/T/ipykernel_57070/2114843666.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  time_features = df.groupby(group_cols).apply(compute_time_features).reset_index()


In [92]:
#50%
print(cloud_50_8.shape)
cloud_50_time = add_calc_cols(cloud_50_8)
cloud_50_8 = cloud_50_8.merge(cloud_50_time, on= ['latitude', 'longitude', 'sample_date', 'cyanobacteria_abundance'])
print(cloud_50_8.shape)

(43184, 18)
(43184, 20)


/var/folders/9z/ltjvr1610w9g431sszz26tr40000gn/T/ipykernel_57070/2114843666.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  time_features = df.groupby(group_cols).apply(compute_time_features).reset_index()


In [93]:
#70% 
print(cloud_70_9.shape)
cloud_70_time = add_calc_cols(cloud_70_9)
cloud_70_9 = cloud_70_9.merge(cloud_70_time, on= ['latitude', 'longitude', 'sample_date', 'cyanobacteria_abundance'])
print(cloud_70_9.shape)

(50580, 18)
(50580, 20)


/var/folders/9z/ltjvr1610w9g431sszz26tr40000gn/T/ipykernel_57070/2114843666.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  time_features = df.groupby(group_cols).apply(compute_time_features).reset_index()


In [94]:
print(cloud_30_7.columns)
print(cloud_50_8.columns)
print(cloud_70_9.columns)

Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo', 'mean_diff_days', 'std_diff_days'],
      dtype='object')
Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo', 'mean_diff_days', 'std_diff_days'],
      dtype='object')
Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo', 'mean_diff_days', 'std_diff_days'],
      dtype='object')


In [95]:
#make everything a csv
#cloud_30_7.to_csv('cloud_30_7.csv', index = False)
#cloud_50_8.to_csv('cloud_50_8.csv', index = False)
#cloud_70_9.to_csv('cloud_70_9.csv', index = False)